# 🐍 Module 5 — Live Coding Patterns
**ml-engineering-drills**

| Section | Content |
|---|---|
| 5.1 | Core patterns — sliding window, two pointers, batch processor |
| 5.2 | Data engineering — grouping, dedup, flatten, inverted index |
| 5.3 | ML patterns — cross-validation, stratified split, metrics, normalization |

---
## 5.1 Core patterns

### Lesson — sliding window patterns

In [ ]:
# ── Sliding window — fixed size ──────────────────────────────
# Key idea: maintain a running sum.
# Add the new element, subtract the one that left the window.
# O(n) instead of O(n*k) for a naive nested loop.

from typing import List
import math
from collections import deque

def max_sum_window_demo(arr: List[float], k: int) -> float:
    """Return the maximum sum of any contiguous subarray of size k."""
    window = sum(arr[:k])
    best   = window
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]   # slide
        best = max(best, window)
    return best

print('max_sum_window([2,1,5,1,3,2], 3):', max_sum_window_demo([2,1,5,1,3,2], 3))  # 9

# ── Sliding window — variable size ───────────────────────────
# Key idea: left pointer advances when condition is violated.
# dict tracks last seen position of each character.

def longest_no_repeat_demo(s: str) -> int:
    """Length of longest substring without repeating characters."""
    seen  = {}
    left  = 0
    best  = 0
    for right, c in enumerate(s):
        if c in seen and seen[c] >= left:
            left = seen[c] + 1   # shrink window
        seen[c] = right
        best = max(best, right - left + 1)
    return best

print("longest_no_repeat('abcabcbb'):", longest_no_repeat_demo("abcabcbb"))  # 3

# ── Rolling stats with deque ──────────────────────────────────
# deque(maxlen=k) automatically drops oldest element.
# Population std: divide by n, not n-1.

def rolling_mean_demo(data: List[float], window: int) -> List[float]:
    q      = deque(maxlen=window)
    result = []
    for val in data:
        q.append(val)
        if len(q) == window:
            result.append(sum(q) / window)
    return result

print('rolling_mean([1..10], 3):', rolling_mean_demo(list(range(1, 11)), 3))

### Lesson — two pointers

In [ ]:
# ── Two pointers on sorted array ─────────────────────────────
# Key idea: start with l=0, r=end. If sum too small → l++.
# If sum too large → r--. O(n) instead of O(n²).

from typing import List, Optional, Tuple

def two_sum_sorted_demo(arr: List[int], target: int) -> Optional[Tuple[int, int]]:
    """Return pair (a, b) summing to target in a SORTED array."""
    l, r = 0, len(arr) - 1
    while l < r:
        s = arr[l] + arr[r]
        if s == target:   return (arr[l], arr[r])
        elif s < target:  l += 1
        else:             r -= 1
    return None

print('two_sum([1,2,3,4,6], 6):', two_sum_sorted_demo([1,2,3,4,6], 6))  # (2,4)

# ── Batch processor ──────────────────────────────────────────
# Key idea: range(0, n, batch_size) gives batch start indices.
# extend() flattens batches into one list.

from typing import TypeVar, Callable
T = TypeVar('T')
R = TypeVar('R')

def batch_process_demo(items: List[T],
                        fn: Callable[[List[T]], List[R]],
                        batch_size: int) -> List[R]:
    result = []
    for i in range(0, len(items), batch_size):
        result.extend(fn(items[i:i + batch_size]))
    return result

out = batch_process_demo(list(range(7)), fn=lambda b: [x**2 for x in b], batch_size=3)
print('batch squares:', out)

# KEY INSIGHT — two pointers requires SORTED input.
# For three_sum: outer loop fixes one element, inner uses two pointers.
# Skip duplicates after finding a triplet to avoid repeated results.

### 🏋️ Exercises 5.1

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — max_sum_window
# ════════════════════════════════════════════════════════
# Write max_sum_window(arr, k) → float
# Returns the maximum sum of any contiguous subarray of size k.
# O(n) — no nested loops.

def max_sum_window(arr: List[float], k: int) -> float:
    pass  # complete here

assert max_sum_window([2, 1, 5, 1, 3, 2], 3) == 9
assert max_sum_window([2, 3, 4, 1, 5], 2)    == 7
assert max_sum_window([1, 1, 1, 1, 1], 5)    == 5
assert max_sum_window([10], 1)               == 10
print("max_sum_window: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def max_sum_window(arr, k):
#     window = sum(arr[:k])
#     best   = window
#     for i in range(k, len(arr)):
#         window += arr[i] - arr[i - k]
#         best = max(best, window)
#     return best

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — longest_no_repeat
# ════════════════════════════════════════════════════════
# Write longest_no_repeat(s: str) → int
# Returns the length of the longest substring without repeating characters.
# O(n) — use a dict to track last seen index.

def longest_no_repeat(s: str) -> int:
    pass  # complete here

assert longest_no_repeat("abcabcbb") == 3
assert longest_no_repeat("bbbbb")    == 1
assert longest_no_repeat("pwwkew")   == 3
assert longest_no_repeat("")         == 0
assert longest_no_repeat("abcd")     == 4
print("longest_no_repeat: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def longest_no_repeat(s):
#     seen = {}
#     left = 0
#     best = 0
#     for right, c in enumerate(s):
#         if c in seen and seen[c] >= left:
#             left = seen[c] + 1
#         seen[c] = right
#         best = max(best, right - left + 1)
#     return best

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — rolling_stats
# ════════════════════════════════════════════════════════
# Write rolling_stats(data, window) → List[dict]
# Returns {'mean': float, 'std': float} for each position
# where the window is full. Use deque with maxlen.
# std = population std (divide by n, not n-1).

def rolling_stats(data: List[float], window: int) -> List[dict]:
    pass  # complete here

result = rolling_stats([2, 4, 6, 8, 10], 3)
assert len(result) == 3
assert result[0]['mean'] == 4.0
assert abs(result[0]['std'] - 1.6329) < 1e-3
assert result[1]['mean'] == 6.0
print("rolling_stats: all tests passed")
print('result:', result)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def rolling_stats(data, window):
#     q      = deque(maxlen=window)
#     result = []
#     for val in data:
#         q.append(val)
#         if len(q) == window:
#             mean = sum(q) / window
#             std  = math.sqrt(sum((x - mean)**2 for x in q) / window)
#             result.append({'mean': round(mean, 4), 'std': round(std, 4)})
#     return result

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 4 — two_sum_sorted + three_sum_sorted
# ════════════════════════════════════════════════════════
# Write two_sum_sorted(arr, target) → tuple | None
# Given a SORTED array, return the pair (a, b) summing to target.
# O(n) — no nested loops, no hash map.
#
# Write three_sum_sorted(arr, target) → List[tuple]
# Returns ALL unique triplets summing to target. O(n²).

def two_sum_sorted(arr: List[int], target: int) -> Optional[Tuple[int, int]]:
    pass  # complete here

def three_sum_sorted(arr: List[int], target: int) -> List[Tuple[int, int, int]]:
    pass  # complete here

assert two_sum_sorted([1, 2, 3, 4, 6], 6)  == (2, 4)
assert two_sum_sorted([1, 2, 3, 4, 6], 10) == (4, 6)
assert two_sum_sorted([1, 2, 3], 10)        is None
print("two_sum_sorted: all tests passed")

result = three_sum_sorted([-3, -1, 0, 1, 2, 3], 0)
assert (-3, 0, 3) in result
assert (-3, 1, 2) in result
assert (-1, 0, 1) in result
print("three_sum_sorted: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def two_sum_sorted(arr, target):
#     l, r = 0, len(arr) - 1
#     while l < r:
#         s = arr[l] + arr[r]
#         if s == target:   return (arr[l], arr[r])
#         elif s < target:  l += 1
#         else:             r -= 1
#     return None
#
# def three_sum_sorted(arr, target):
#     results = []
#     for i in range(len(arr) - 2):
#         if i > 0 and arr[i] == arr[i-1]: continue
#         l, r = i + 1, len(arr) - 1
#         while l < r:
#             s = arr[i] + arr[l] + arr[r]
#             if s == target:
#                 results.append((arr[i], arr[l], arr[r]))
#                 while l < r and arr[l] == arr[l+1]: l += 1
#                 while l < r and arr[r] == arr[r-1]: r -= 1
#                 l += 1; r -= 1
#             elif s < target: l += 1
#             else:            r -= 1
#     return results

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 5 — batch_process + batch_process_with_progress
# ════════════════════════════════════════════════════════
# Write batch_process(items, fn, batch_size) → List
# Splits items into batches, applies fn, returns flat list.
# Last batch may be smaller than batch_size.
#
# Write batch_process_with_progress(items, fn, batch_size)
# Same but prints 'batch {i}/{total}: {n} items' for each batch.

def batch_process(items: List[T],
                  fn: Callable[[List[T]], List[R]],
                  batch_size: int) -> List[R]:
    pass  # complete here

def batch_process_with_progress(items: List[T],
                                 fn: Callable[[List[T]], List[R]],
                                 batch_size: int) -> List[R]:
    pass  # complete here

result = batch_process(list(range(10)), fn=lambda b: [x**2 for x in b], batch_size=3)
assert result == [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
print("batch_process: all tests passed")

print('\nWith progress:')
result2 = batch_process_with_progress(list(range(7)), fn=lambda b: [x*2 for x in b], batch_size=3)
assert result2 == [0, 2, 4, 6, 8, 10, 12]
print("batch_process_with_progress: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def batch_process(items, fn, batch_size):
#     result = []
#     for i in range(0, len(items), batch_size):
#         result.extend(fn(items[i:i + batch_size]))
#     return result
#
# def batch_process_with_progress(items, fn, batch_size):
#     result = []
#     total  = (len(items) + batch_size - 1) // batch_size
#     for i, start in enumerate(range(0, len(items), batch_size), 1):
#         batch = items[start:start + batch_size]
#         print(f'batch {i}/{total}: {len(batch)} items')
#         result.extend(fn(batch))
#     return result

---
## 5.2 Data engineering patterns

### Lesson — grouping, deduplication, flatten

In [ ]:
from collections import defaultdict
from typing import List, Dict, Callable, TypeVar, Any
import string
T = TypeVar('T')

# ── group_by ─────────────────────────────────────────────────
# defaultdict(list) avoids KeyError on first insertion.
# key_fn extracts the grouping key from each record.

def group_by_demo(records: List[T], key_fn: Callable) -> Dict:
    groups = defaultdict(list)
    for r in records:
        groups[key_fn(r)].append(r)
    return dict(groups)

data = [('IDF', 100), ('PACA', 80), ('IDF', 120), ('PACA', 70)]
print('grouped:', group_by_demo(data, lambda r: r[0]))

# ── deduplicate preserving order ──────────────────────────────
# set() for O(1) lookup, list for ordered output.
# dict.fromkeys() is a one-liner alternative for hashable items.

def deduplicate_demo(items: list) -> list:
    seen, result = set(), []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

print('dedup:', deduplicate_demo([1, 2, 2, 3, 1, 4]))
print('one-liner:', list(dict.fromkeys([1, 2, 2, 3, 1, 4])))

# ── flatten arbitrary depth ───────────────────────────────────
# Recursion: if item is a list, recurse; else append.
# For depth-limited: pass depth counter, stop when 0.

def flatten_demo(nested: list) -> list:
    result = []
    for item in nested:
        if isinstance(item, list): result.extend(flatten_demo(item))
        else:                      result.append(item)
    return result

print('flatten:', flatten_demo([1, [2, [3, [4]]], 5]))

# ── inverted index ────────────────────────────────────────────
# dict.fromkeys(words) deduplicates words within a document.
# search: set intersection across posting lists.

def build_index_demo(docs: List[str]) -> Dict:
    index = defaultdict(list)
    for i, doc in enumerate(docs):
        words = doc.lower().translate(str.maketrans('','',string.punctuation)).split()
        for word in dict.fromkeys(words):
            index[word].append(i)
    return dict(index)

idx = build_index_demo(['ml is great', 'deep learning', 'ml learning'])
print('index[ml]:', idx.get('ml'))
print('search ml+learning:', sorted(set(idx.get('ml',[])) & set(idx.get('learning',[]))))

### 🏋️ Exercises 5.2

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — group_by + group_and_aggregate
# ════════════════════════════════════════════════════════
# Write group_by(records, key_fn) → dict
# Groups records by key_fn result. Returns {key: [records]}.
# Use defaultdict — do NOT use itertools.groupby.
#
# Write group_and_aggregate(records, key_fn, agg_fn) → dict
# Same grouping, applies agg_fn(list_of_records) → scalar per group.

def group_by(records: List[T], key_fn: Callable[[T], str]) -> Dict[str, List[T]]:
    pass  # complete here

def group_and_aggregate(records: List[T],
                         key_fn: Callable[[T], str],
                         agg_fn: Callable[[List[T]], Any]) -> Dict[str, Any]:
    pass  # complete here

records = [
    {'region': 'IDF',  'value': 100.0},
    {'region': 'PACA', 'value': 80.0},
    {'region': 'IDF',  'value': 120.0},
    {'region': 'AURA', 'value': 90.0},
    {'region': 'PACA', 'value': 70.0},
    {'region': 'IDF',  'value': 110.0},
]

grouped = group_by(records, key_fn=lambda r: r['region'])
assert len(grouped['IDF'])  == 3
assert len(grouped['PACA']) == 2
print("group_by: all tests passed")

agg = group_and_aggregate(
    records,
    key_fn=lambda r: r['region'],
    agg_fn=lambda g: round(sum(r['value'] for r in g) / len(g), 2),
)
assert agg['IDF']  == 110.0
assert agg['PACA'] == 75.0
assert agg['AURA'] == 90.0
print("group_and_aggregate: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def group_by(records, key_fn):
#     groups = defaultdict(list)
#     for r in records:
#         groups[key_fn(r)].append(r)
#     return dict(groups)
#
# def group_and_aggregate(records, key_fn, agg_fn):
#     groups = group_by(records, key_fn)
#     return {k: agg_fn(v) for k, v in groups.items()}

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — deduplicate + deduplicate_by
# ════════════════════════════════════════════════════════
# Write deduplicate(items) → list
# Removes duplicates while PRESERVING insertion order.
# Do NOT convert to set (loses order).
#
# Write deduplicate_by(records, key_fn) → list
# Deduplicates records by key function. Keeps FIRST occurrence.

def deduplicate(items: List[T]) -> List[T]:
    pass  # complete here

def deduplicate_by(records: List[T], key_fn: Callable[[T], Any]) -> List[T]:
    pass  # complete here

assert deduplicate([1, 2, 2, 3, 1, 4])      == [1, 2, 3, 4]
assert deduplicate(['a', 'b', 'a', 'c'])     == ['a', 'b', 'c']
assert deduplicate([])                        == []
print("deduplicate: all tests passed")

records = [
    {'id': 1, 'name': 'Alice', 'score': 0.9},
    {'id': 2, 'name': 'Bob',   'score': 0.8},
    {'id': 1, 'name': 'Alice', 'score': 0.95},
    {'id': 3, 'name': 'Charlie', 'score': 0.7},
]
deduped = deduplicate_by(records, key_fn=lambda r: r['id'])
assert len(deduped) == 3
assert deduped[0]['score'] == 0.9
print("deduplicate_by: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def deduplicate(items):
#     seen, result = set(), []
#     for item in items:
#         if item not in seen:
#             seen.add(item)
#             result.append(item)
#     return result
#
# def deduplicate_by(records, key_fn):
#     seen, result = set(), []
#     for r in records:
#         k = key_fn(r)
#         if k not in seen:
#             seen.add(k)
#             result.append(r)
#     return result

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — flatten + flatten_depth
# ════════════════════════════════════════════════════════
# Write flatten(nested) → list
# Flattens an arbitrarily nested list. Use recursion.
#
# Write flatten_depth(nested, depth) → list
# Same but only flattens up to `depth` levels.

def flatten(nested: list) -> list:
    pass  # complete here

def flatten_depth(nested: list, depth: int) -> list:
    pass  # complete here

assert flatten([1, [2, 3], [4, [5, 6]]]) == [1, 2, 3, 4, 5, 6]
assert flatten([[1, [2]], [3, [4, [5]]]]) == [1, 2, 3, 4, 5]
assert flatten([]) == []
print("flatten: all tests passed")

nested = [1, [2, [3, [4]]]]
assert flatten_depth(nested, 0) == [1, [2, [3, [4]]]]
assert flatten_depth(nested, 1) == [1, 2, [3, [4]]]
assert flatten_depth(nested, 2) == [1, 2, 3, [4]]
assert flatten_depth(nested, 3) == [1, 2, 3, 4]
print("flatten_depth: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def flatten(nested):
#     result = []
#     for item in nested:
#         if isinstance(item, list): result.extend(flatten(item))
#         else:                      result.append(item)
#     return result
#
# def flatten_depth(nested, depth):
#     if depth == 0: return nested
#     result = []
#     for item in nested:
#         if isinstance(item, list): result.extend(flatten_depth(item, depth - 1))
#         else:                      result.append(item)
#     return result

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 4 — build_inverted_index + search
# ════════════════════════════════════════════════════════
# Write build_inverted_index(documents) → dict
# {word: [doc indices where word appears]} — lowercased, no punct.
# Each index appears only once per word.
#
# Write search(index, query) → List[int]
# Returns doc indices containing ALL words in query.

def build_inverted_index(documents: List[str]) -> Dict[str, List[int]]:
    pass  # complete here

def search(index: Dict[str, List[int]], query: str) -> List[int]:
    pass  # complete here

docs = [
    'machine learning is powerful',
    'deep learning needs data',
    'machine learning needs data too',
    'data science and machine learning',
]
index = build_inverted_index(docs)
assert index['machine']  == [0, 2, 3]
assert index['learning'] == [0, 1, 2, 3]
assert index['data']     == [1, 2, 3]
print("build_inverted_index: all tests passed")

assert search(index, 'machine learning') == [0, 2, 3]
assert search(index, 'deep data')        == [1]
assert search(index, 'quantum')          == []
print("search: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def build_inverted_index(documents):
#     index = defaultdict(list)
#     for i, doc in enumerate(documents):
#         words = doc.lower().translate(
#             str.maketrans('', '', string.punctuation)).split()
#         for word in dict.fromkeys(words):
#             index[word].append(i)
#     return dict(index)
#
# def search(index, query):
#     words   = query.lower().split()
#     results = [set(index.get(w, [])) for w in words]
#     if not results: return []
#     return sorted(set.intersection(*results))

---
## 5.3 ML patterns

### Lesson — cross-validation and metrics from scratch

In [ ]:
import numpy as np
import copy
from typing import Callable, Dict, Any

# ── k-fold cross-validation ───────────────────────────────────
# Key steps:
# 1. Shuffle indices with np.random.permutation
# 2. Split into k folds with np.array_split
# 3. For fold i: val=folds[i], train=all other folds concatenated
# 4. ALWAYS deepcopy the model — never reuse a fitted instance
# 5. Use predict_proba for AUC, predict for accuracy/f1

def kfold_cv_demo(X, y, model, k, metric_fn):
    n       = len(X)
    indices = np.random.permutation(n)
    folds   = np.array_split(indices, k)
    scores  = []
    for i in range(k):
        val_idx   = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        m = copy.deepcopy(model)
        m.fit(X[train_idx], y[train_idx])
        preds = m.predict_proba(X[val_idx])[:, 1] if hasattr(m, 'predict_proba') else m.predict(X[val_idx])
        scores.append(metric_fn(y[val_idx], preds))
    scores = np.array(scores)
    return {'scores': scores.tolist(), 'mean': float(scores.mean()), 'std': float(scores.std())}

# ── metrics from scratch ──────────────────────────────────────
# All derived from TP, FP, FN, TN counts.
# ROC-AUC: sort by score desc, compute TPR/FPR at each threshold,
#          area under curve via trapezoidal rule.

def precision_demo(y_true, y_pred):
    tp = ((y_pred==1) & (y_true==1)).sum()
    fp = ((y_pred==1) & (y_true==0)).sum()
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def roc_auc_demo(y_true, y_scores):
    order  = np.argsort(y_scores)[::-1]
    y_true = np.array(y_true)[order]
    tps    = np.cumsum(y_true)
    fps    = np.cumsum(1 - y_true)
    tpr    = tps / tps[-1]
    fpr    = fps / fps[-1]
    return float(np.trapz(tpr, fpr))

y_true = np.array([1, 0, 1, 1, 0])
y_pred = np.array([1, 0, 1, 0, 1])
print(f'precision: {precision_demo(y_true, y_pred):.3f}')
y_scores = np.array([0.9, 0.2, 0.8, 0.7, 0.4])
print(f'roc_auc  : {roc_auc_demo(y_true, y_scores):.3f}')

### Lesson — normalization and permutation importance

In [ ]:
import numpy as np

# ── Normalizers: fit/transform pattern ───────────────────────
# fit() stores parameters learned on train set (trailing underscore).
# transform() applies them — never refit on test data.

class ZScoreNormalizerDemo:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0)
        return self
    def transform(self, X):
        return (X - self.mean_) / np.where(self.std_ == 0, 1, self.std_)

X_train = np.array([[1., 10.], [2., 20.], [3., 30.]])
X_test  = np.array([[4., 40.], [5., 50.]])
norm = ZScoreNormalizerDemo().fit(X_train)
print('z-score train mean:', norm.transform(X_train).mean(axis=0).round(6))
print('z-score test      :', norm.transform(X_test).round(3))

# ── Permutation importance ────────────────────────────────────
# Shuffle one column at a time, measure score drop.
# importance = baseline_score - mean(shuffled_scores).
# Higher = feature matters more.

def permutation_importance_demo(model, X, y, metric_fn, n_repeats=3):
    X        = np.array(X, dtype=float)
    preds    = model.predict_proba(X)[:,1] if hasattr(model,'predict_proba') else model.predict(X)
    baseline = metric_fn(y, preds)
    rng      = np.random.default_rng(42)
    importances = np.zeros(X.shape[1])
    for col in range(X.shape[1]):
        scores = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[:, col] = rng.permutation(X_perm[:, col])
            p = model.predict_proba(X_perm)[:,1] if hasattr(model,'predict_proba') else model.predict(X_perm)
            scores.append(metric_fn(y, p))
        importances[col] = baseline - np.mean(scores)
    return {'importances': importances, 'baseline': baseline}

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
X, y = load_iris(return_X_y=True)
y_bin = (y == 0).astype(int)
model = RandomForestClassifier(n_estimators=20, random_state=42).fit(X, y_bin)
result = permutation_importance_demo(model, X, y_bin, roc_auc_score, n_repeats=3)
print(f'baseline AUC: {result["baseline"]:.4f}')
print(f'importances : {result["importances"].round(4)}')

### 🏋️ Exercises 5.3

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — kfold_cv from scratch
# ════════════════════════════════════════════════════════
# Write kfold_cv(X, y, model, k, metric_fn) → dict
# Manual k-fold cross-validation.
# - Shuffles indices before splitting
# - Fits model on train fold, predicts on val fold
# - Applies metric_fn(y_true, y_pred) → float
# - Returns {'scores': List[float], 'mean': float, 'std': float}
# CRITICAL: no data leakage — deepcopy model, fit only on train fold.

import numpy as np
import copy
from typing import Callable, Dict, Any

def kfold_cv(X: np.ndarray, y: np.ndarray,
             model: Any, k: int,
             metric_fn: Callable) -> Dict:
    pass  # complete here

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X    = StandardScaler().fit_transform(X)

result = kfold_cv(
    X, y,
    model=LogisticRegression(max_iter=1000, random_state=42),
    k=5,
    metric_fn=roc_auc_score,
)
if result:
    print(f"scores : {[round(s, 4) for s in result['scores']]}")
    print(f"mean   : {result['mean']:.4f}")
    print(f"std    : {result['std']:.4f}")
    assert len(result['scores']) == 5
    assert 0.95 < result['mean'] < 1.0
    print("kfold_cv: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def kfold_cv(X, y, model, k, metric_fn):
#     n       = len(X)
#     indices = np.random.permutation(n)
#     folds   = np.array_split(indices, k)
#     scores  = []
#     for i in range(k):
#         val_idx   = folds[i]
#         train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
#         m = copy.deepcopy(model)
#         m.fit(X[train_idx], y[train_idx])
#         preds = m.predict_proba(X[val_idx])[:, 1] if hasattr(m, 'predict_proba') else m.predict(X[val_idx])
#         scores.append(metric_fn(y[val_idx], preds))
#     scores = np.array(scores)
#     return {'scores': scores.tolist(), 'mean': float(scores.mean()), 'std': float(scores.std())}

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — stratified_split from scratch
# ════════════════════════════════════════════════════════
# Write stratified_split(X, y, test_size, random_state) → tuple
# Returns (X_train, X_test, y_train, y_test).
# Class distribution in train and test must match original.
# Do NOT use sklearn — implement from scratch.

import numpy as np
from typing import Tuple

def stratified_split(X: np.ndarray, y: np.ndarray,
                      test_size: float = 0.2,
                      random_state: int = 42) -> Tuple:
    pass  # complete here

np.random.seed(0)
X = np.random.randn(1000, 5)
y = np.array([0]*800 + [1]*200)

X_tr, X_te, y_tr, y_te = stratified_split(X, y, test_size=0.2, random_state=42)
assert len(X_tr) == 800
assert len(X_te) == 200
assert abs(y_te.mean() - 0.2) < 0.02
assert abs(y_tr.mean() - 0.2) < 0.02
print("stratified_split: all tests passed")
print(f"train ratio: {y_tr.mean():.3f} | test ratio: {y_te.mean():.3f}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def stratified_split(X, y, test_size=0.2, random_state=42):
#     rng     = np.random.default_rng(random_state)
#     classes = np.unique(y)
#     tr_idx, te_idx = [], []
#     for cls in classes:
#         idx    = rng.permutation(np.where(y == cls)[0])
#         n_test = max(1, int(len(idx) * test_size))
#         te_idx.extend(idx[:n_test])
#         tr_idx.extend(idx[n_test:])
#     return X[tr_idx], X[te_idx], y[tr_idx], y[te_idx]

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — 6 metrics from scratch
# ════════════════════════════════════════════════════════
# Implement these 6 metrics using only numpy. No sklearn.
#
# accuracy(y_true, y_pred) → float
# precision(y_true, y_pred) → float   TP / (TP + FP)
# recall(y_true, y_pred) → float      TP / (TP + FN)
# f1(y_true, y_pred) → float
# confusion_matrix(y_true, y_pred) → np.ndarray (2, 2)
# roc_auc(y_true, y_scores) → float   trapezoidal rule

import numpy as np

def accuracy(y_true, y_pred): pass
def precision(y_true, y_pred): pass
def recall(y_true, y_pred): pass
def f1(y_true, y_pred): pass
def confusion_matrix(y_true, y_pred): pass
def roc_auc(y_true, y_scores): pass

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
from sklearn.metrics import confusion_matrix as sk_cm

np.random.seed(42)
y_true   = np.array([1,1,0,1,0,0,1,0,1,1,0,1,0,0,1])
y_pred   = np.array([1,0,0,1,1,0,1,0,0,1,0,1,1,0,1])
y_scores = np.random.rand(15)

tol = 1e-6
assert abs(accuracy(y_true, y_pred)  - accuracy_score(y_true, y_pred))  < tol
assert abs(precision(y_true, y_pred) - precision_score(y_true, y_pred)) < tol
assert abs(recall(y_true, y_pred)    - recall_score(y_true, y_pred))    < tol
assert abs(f1(y_true, y_pred)        - f1_score(y_true, y_pred))        < tol
assert np.allclose(confusion_matrix(y_true, y_pred), sk_cm(y_true, y_pred))
assert abs(roc_auc(y_true, y_scores) - roc_auc_score(y_true, y_scores)) < tol
print("all metrics: tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def accuracy(y_true, y_pred):
#     return (np.array(y_true) == np.array(y_pred)).mean()
#
# def precision(y_true, y_pred):
#     tp = ((y_pred==1) & (y_true==1)).sum()
#     fp = ((y_pred==1) & (y_true==0)).sum()
#     return tp / (tp + fp) if (tp + fp) > 0 else 0.0
#
# def recall(y_true, y_pred):
#     tp = ((y_pred==1) & (y_true==1)).sum()
#     fn = ((y_pred==0) & (y_true==1)).sum()
#     return tp / (tp + fn) if (tp + fn) > 0 else 0.0
#
# def f1(y_true, y_pred):
#     p, r = precision(y_true, y_pred), recall(y_true, y_pred)
#     return 2 * p * r / (p + r) if (p + r) > 0 else 0.0
#
# def confusion_matrix(y_true, y_pred):
#     tn = ((y_pred==0) & (y_true==0)).sum()
#     fp = ((y_pred==1) & (y_true==0)).sum()
#     fn = ((y_pred==0) & (y_true==1)).sum()
#     tp = ((y_pred==1) & (y_true==1)).sum()
#     return np.array([[tn, fp], [fn, tp]])
#
# def roc_auc(y_true, y_scores):
#     order  = np.argsort(y_scores)[::-1]
#     y_true = np.array(y_true)[order]
#     tps    = np.cumsum(y_true)
#     fps    = np.cumsum(1 - y_true)
#     tpr    = tps / tps[-1]
#     fpr    = fps / fps[-1]
#     return float(np.trapz(tpr, fpr))

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 4 — 4 normalizers from scratch
# ════════════════════════════════════════════════════════
# Implement as classes with fit/transform.
# fit() learns from train, transform() applies — numpy only.
#
# ZScoreNormalizer  → (x - mean) / std  per column
# MinMaxNormalizer  → (x - min) / (max - min) per column
# RobustNormalizer  → (x - median) / IQR per column
# L2Normalizer      → x / ||x||₂  per ROW (stateless)

import numpy as np

class ZScoreNormalizer:
    def fit(self, X): pass
    def transform(self, X): pass

class MinMaxNormalizer:
    def fit(self, X): pass
    def transform(self, X): pass

class RobustNormalizer:
    def fit(self, X): pass
    def transform(self, X): pass

class L2Normalizer:
    def fit(self, X): return self
    def transform(self, X): pass

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer

np.random.seed(42)
X_train = np.random.randn(100, 4) * np.array([1, 10, 100, 1000])
X_test  = np.random.randn(20, 4)  * np.array([1, 10, 100, 1000])

for MyClass, SkClass in [
    (ZScoreNormalizer, StandardScaler),
    (MinMaxNormalizer, MinMaxScaler),
    (RobustNormalizer, RobustScaler),
]:
    my = MyClass(); my.fit(X_train)
    sk = SkClass().fit(X_train)
    out = my.transform(X_test)
    if out is not None:
        assert np.allclose(out, sk.transform(X_test), atol=1e-6), f'{MyClass.__name__} failed'
        print(f'{MyClass.__name__}: passed')

my_l2 = L2Normalizer(); my_l2.fit(X_train)
out = my_l2.transform(X_test)
if out is not None:
    assert np.allclose(out, Normalizer(norm='l2').transform(X_test), atol=1e-6)
    print('L2Normalizer: passed')

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# class ZScoreNormalizer:
#     def fit(self, X):
#         self.mean_ = X.mean(axis=0); self.std_ = X.std(axis=0); return self
#     def transform(self, X):
#         return (X - self.mean_) / np.where(self.std_ == 0, 1, self.std_)
#
# class MinMaxNormalizer:
#     def fit(self, X):
#         self.min_ = X.min(axis=0); self.max_ = X.max(axis=0); return self
#     def transform(self, X):
#         rng = self.max_ - self.min_
#         return (X - self.min_) / np.where(rng == 0, 1, rng)
#
# class RobustNormalizer:
#     def fit(self, X):
#         self.median_ = np.median(X, axis=0)
#         self.iqr_    = np.percentile(X, 75, axis=0) - np.percentile(X, 25, axis=0)
#         return self
#     def transform(self, X):
#         return (X - self.median_) / np.where(self.iqr_ == 0, 1, self.iqr_)
#
# class L2Normalizer:
#     def fit(self, X): return self
#     def transform(self, X):
#         norms = np.linalg.norm(X, axis=1, keepdims=True)
#         return X / np.where(norms == 0, 1, norms)

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 5 — permutation_importance
# ════════════════════════════════════════════════════════
# Write permutation_importance(model, X, y, metric_fn, n_repeats) → dict
# For each feature:
#   1. Record baseline score on (X, y)
#   2. Shuffle that column n_repeats times, score each time
#   3. importance = baseline - mean(shuffled scores)
# Returns {'importances': np.ndarray (n_features,), 'baseline': float}

import numpy as np
from typing import Callable, Any, Dict

def permutation_importance(model: Any, X: np.ndarray, y: np.ndarray,
                            metric_fn: Callable, n_repeats: int = 5) -> Dict:
    pass  # complete here

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y   = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr   = StandardScaler().fit_transform(X_tr)
X_te   = StandardScaler().fit_transform(X_te)
model  = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr, y_tr)

result = permutation_importance(model, X_te, y_te, metric_fn=roc_auc_score, n_repeats=5)
if result:
    assert 'importances' in result
    assert len(result['importances']) == X.shape[1]
    assert result['baseline'] > 0.95
    top5 = np.argsort(result['importances'])[-5:][::-1]
    print(f"baseline AUC : {result['baseline']:.4f}")
    print(f"top 5 features: {list(top5)}")
    print(f"importances   : {result['importances'][top5].round(4)}")
    print("permutation_importance: all tests passed")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def permutation_importance(model, X, y, metric_fn, n_repeats=5):
#     X        = np.array(X, dtype=float)
#     preds    = model.predict_proba(X)[:,1] if hasattr(model,'predict_proba') else model.predict(X)
#     baseline = metric_fn(y, preds)
#     rng      = np.random.default_rng(42)
#     importances = np.zeros(X.shape[1])
#     for col in range(X.shape[1]):
#         scores = []
#         for _ in range(n_repeats):
#             X_perm = X.copy()
#             X_perm[:, col] = rng.permutation(X_perm[:, col])
#             p = model.predict_proba(X_perm)[:,1] if hasattr(model,'predict_proba') else model.predict(X_perm)
#             scores.append(metric_fn(y, p))
#         importances[col] = baseline - np.mean(scores)
#     return {'importances': importances, 'baseline': baseline}

---
## 📋 Module 5 Recap

| Topic | Test |
|---|---|
| `max_sum_window` | O(n) — running sum, no nested loop |
| `longest_no_repeat` | dict tracking last seen index, left pointer |
| `rolling_stats` | deque with maxlen, population std |
| `two_sum_sorted` | two pointers, O(n), sorted input only |
| `three_sum_sorted` | outer loop + two pointers, O(n²) |
| `batch_process` | extend result, last batch may be smaller |
| `group_by` | defaultdict(list), key function |
| `group_and_aggregate` | group_by + dict comprehension with agg_fn |
| `deduplicate` | seen set + preserve order |
| `deduplicate_by` | seen set on key_fn result |
| `flatten` | recursion on isinstance(item, list) |
| `flatten_depth` | same + depth counter |
| `build_inverted_index` | defaultdict, dict.fromkeys for dedup |
| `search` | set intersection across posting lists |
| `kfold_cv` | deepcopy model, no leakage, metric_fn |
| `stratified_split` | split per class, concatenate |
| metrics (6) | TP/FP/FN/TN, trapezoidal rule for AUC |
| normalizers (4) | fit stores params, transform applies them |
| `permutation_importance` | shuffle one column at a time, n_repeats |
